# House Price Prediction

**Goal:** Predict residential sale prices from property characteristics using the Ames Housing dataset.

**Data source:** Ames Housing dataset via `sklearn.datasets.fetch_openml` (OpenML ID 42165).

This notebook covers: EDA -> cleaning & feature engineering -> baseline vs. stronger models -> evaluation -> interpretation.

> Fill in the *italic* prompts throughout with your own observations as you run each cell — that narrative is what makes this project stand out.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 100)

## 1. Load the data

In [ ]:
housing = fetch_openml(name="house_prices", as_frame=True, parser="auto")
df = housing.frame.copy()

print(df.shape)
df.head()

In [ ]:
df.info()
print("\nMissing values (top 15):")
print(df.isnull().sum().sort_values(ascending=False).head(15))

*What do you notice about the size, dtypes, and missingness of this dataset? Any columns that are almost entirely missing (candidates to drop)?*

## 2. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["SalePrice"], kde=True)
plt.title("Distribution of Sale Price")
plt.show()

print("Skew:", df["SalePrice"].skew())

*Is SalePrice skewed? If so, a log transform of the target often helps linear models — note your decision here.*

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
top_corr = numeric_df.corr()["SalePrice"].sort_values(ascending=False).head(11)
print(top_corr)

plt.figure(figsize=(8, 6))
sns.heatmap(numeric_df[top_corr.index].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap: top features vs. SalePrice")
plt.show()

*Which features correlate most strongly with SalePrice? Do these make intuitive sense? Flag any surprises.*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["GrLivArea", "OverallQual", "YearBuilt"]):
    sns.scatterplot(data=df, x=col, y="SalePrice", ax=ax, alpha=0.4)
    ax.set_title(f"SalePrice vs {col}")
plt.tight_layout()
plt.show()

*Any visible outliers (e.g. very large GrLivArea with unexpectedly low price)? Decide whether to drop or keep them, and say why.*

## 3. Cleaning & Feature Engineering

In [ ]:
# Drop columns that are missing for the vast majority of rows
missing_frac = df.isnull().mean()
cols_to_drop = missing_frac[missing_frac > 0.8].index.tolist()
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>80% missing): {cols_to_drop}")

In [ ]:
# Example feature engineering — extend this with your own domain reasoning
df["HouseAge"] = df["YrSold"].astype(float) - df["YearBuilt"].astype(float)
df["TotalSF"] = df["TotalBsmtSF"].fillna(0) + df["1stFlrSF"] + df["2ndFlrSF"]
df["PricePerSF"] = df["SalePrice"] / df["TotalSF"].replace(0, np.nan)

df[["HouseAge", "TotalSF", "PricePerSF"]].describe()

*Explain each engineered feature in a sentence: what does it capture that the raw columns didn't?*

## 4. Train/Test Split & Preprocessing Pipeline

Splitting **before** fitting any imputer/encoder/scaler to avoid data leakage.

In [ ]:
target = "SalePrice"
features = [c for c in df.columns if c not in [target, "PricePerSF", "Id"]]

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## 5. Baseline Model: Linear Regression

In [ ]:
lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)

lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2 = r2_score(y_test, lr_preds)
print(f"Linear Regression -> RMSE: ${lr_rmse:,.0f} | R2: {lr_r2:.3f}")

## 6. Comparison Models: Random Forest & XGBoost

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))
])
rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2 = r2_score(y_test, rf_preds)
print(f"Random Forest -> RMSE: ${rf_rmse:,.0f} | R2: {rf_r2:.3f}")

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, random_state=42))
])
xgb_pipeline.fit(X_train, y_train)
xgb_preds = xgb_pipeline.predict(X_test)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_r2 = r2_score(y_test, xgb_preds)
print(f"XGBoost -> RMSE: ${xgb_rmse:,.0f} | R2: {xgb_r2:.3f}")

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "XGBoost"],
    "RMSE": [lr_rmse, rf_rmse, xgb_rmse],
    "R2": [lr_r2, rf_r2, xgb_r2]
})
results

*Which model performed best? Talk about the tradeoff: Linear Regression is interpretable but likely underfits non-linear relationships; tree-based models usually score better but are harder to explain directly.*

## 7. Cross-Validation (sanity check on the split)

In [ ]:
cv_scores = cross_val_score(xgb_pipeline, X, y, cv=5, scoring="neg_root_mean_squared_error")
print("XGBoost 5-fold CV RMSE:", -cv_scores.mean(), "+/-", cv_scores.std())

## 8. Feature Importance

In [ ]:
feature_names = xgb_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = xgb_pipeline.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})
importance_df = importance_df.sort_values("importance", ascending=False).head(15)

plt.figure(figsize=(8, 6))
sns.barplot(data=importance_df, x="importance", y="feature")
plt.title("Top 15 Feature Importances (XGBoost)")
plt.tight_layout()
plt.show()

*In plain language: what are the top drivers of price according to the model? Does this match your EDA findings from Section 2?*

## 9. Conclusion

*Summarize in 3-5 sentences for a non-technical reader:*
- *What did you set out to predict?*
- *Which model worked best and how accurate is it (in dollar terms)?*
- *What drives price the most?*
- *What are the limitations, and what would you try next given more time?*